# 처음부터 LBSTER 모델 훈련하기

이 튜토리얼에서는 자체 데이터셋을 사용하여 처음부터 LBSTER 단백질 언어 모델을 훈련하는 방법을 보여줍니다. 데이터 준비, 모델 구성, 훈련 및 평가를 다룹니다.

## 설정 및 설치

먼저 LBSTER가 설치되어 있는지 확인하십시오:

```bash
pip install -e .
```

필요한 라이브러리를 가져옵니다:

In [ ]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
import hydra
from omegaconf import DictConfig, OmegaConf
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

In [ ]:
from lobster.data import DataFrameLightningDataModule
from lobster.tokenization import PmlmTokenizerTransform

## 1단계: 데이터 준비

훈련을 위해 단백질 서열이 필요합니다. 이 튜토리얼에서는 작은 합성 데이터셋을 생성합니다:

In [ ]:
# Create a synthetic dataset of protein sequences
def generate_synthetic_data(n_samples=1000, min_len=50, max_len=200):
    """Generate synthetic protein sequences for demonstration."""
    import random
    
    # Standard amino acids
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    
    sequences = []
    for _ in range(n_samples):
        # Random sequence length
        length = random.randint(min_len, max_len)
        # Generate random sequence
        seq = ''.join(random.choice(amino_acids) for _ in range(length))
        sequences.append(seq)
    
    return pd.DataFrame({'sequence': sequences})

In [ ]:
# Generate synthetic data
df = generate_synthetic_data(n_samples=1000)
print(f"Generated {len(df)} protein sequences")
print(f"Example sequence: {df['sequence'].iloc[0]}")

In [ ]:
# Save to CSV for use with hydra config
df.to_csv('synthetic_proteins.csv', index=False)

## 2단계: 구성 파일 만들기

LBSTER는 구성을 위해 Hydra를 사용합니다. 기본 구성을 만들어 보겠습니다:

In [ ]:
# Write a simple training config to a file
train_config = """
# @package _global_

# paths config
paths:
  root_dir: "./"

# setup config 
setup:
  _target_: hydra.utils.instantiate_at_run
  
# data config
data:
  _target_: lobster.data.DataFrameLightningDataModule
  data: 
    _target_: pandas.read_csv
    filepath_or_buffer: synthetic_proteins.csv
  columns: ["sequence"]
  batch_size: 8
  max_length: 256
  num_workers: 2
  transform_fn:
    _target_: lobster.tokenization.PmlmTokenizerTransform
    padding: "max_length"
    truncation: true
    max_length: 256
    tokenizer_dir: "pmlm_tokenizer"

# model config
model:
  _target_: lobster.model.LobsterPMLM
  max_length: 256
  hidden_dim: 128
  n_layer: 2
  n_head: 4
  dim_feedforward: 512
  learning_rate: 1e-4
  weight_decay: 0.01
  warmup_steps: 100

# trainer config
trainer:
  _target_: lightning.pytorch.Trainer
  max_epochs: 5
  accelerator: auto
  devices: 1
  log_every_n_steps: 10
  val_check_interval: 0.5
  
# logger config
logger:
  _target_: lightning.pytorch.loggers.TensorBoardLogger
  save_dir: "./logs"
  name: "lobster-training"
  
# callbacks config
callbacks:
  checkpoint_callback:
    _target_: lightning.pytorch.callbacks.ModelCheckpoint
    monitor: "val_loss"
    mode: "min"
    save_top_k: 1
    save_last: true
    dirpath: "./checkpoints"
    filename: "{epoch}-{val_loss:.2f}"
  early_stopping:
    _target_: lightning.pytorch.callbacks.EarlyStopping
    monitor: "val_loss"
    patience: 3
    mode: "min"
    
# don't actually run training in this example
dryrun: true

# don't test after training
run_test: false
"""

In [ ]:
# Save the config to a YAML file
with open('train_config.yaml', 'w') as f:
    f.write(train_config)

## 3단계: 모델 구성

모델 매개변수를 구성하는 방법을 이해해 보겠습니다:

In [ ]:
# Print important model configuration parameters
print("Key model configuration parameters:")
print("-" * 40)
print("hidden_dim: Size of the hidden layers")
print("n_layer: Number of transformer layers")
print("n_head: Number of attention heads")
print("dim_feedforward: Size of the feedforward network in transformer layers")
print("learning_rate: Initial learning rate for optimizer")
print("weight_decay: L2 regularization")
print("warmup_steps: Number of warmup steps for learning rate scheduler")
print("-" * 40)

## 4단계: 훈련 과정

hydra CLI를 사용하여 모델을 훈련하지만, 이 노트북에서는 프로세스를 설명만 하겠습니다:

In [ ]:
# Here's how you would execute the training from the command line:
print("To train the model, run this command from the terminal:")
print("lobster_train -cn train_config")
print("\nFor larger datasets or more complex models, you might want to add more options:")
print("lobster_train -cn train_config trainer.max_epochs=10 trainer.devices=2 model.learning_rate=5e-5")

## 5단계: 시뮬레이션된 훈련 과정

프로세스를 이해하기 위해 훈련 중에 일어나는 일을 시뮬레이션해 보겠습니다:

In [ ]:
# Simple simulation of training process (not actually training)
def simulate_training_process():
    print("\nSimulating training process:")
    print("1. Loading configuration...")
    print("2. Preparing data module...")
    print("3. Initializing model...")
    print("4. Setting up trainer with callbacks...")
    
    # Simple training loss simulation
    epochs = 5
    train_losses = [2.5 - i * 0.4 + 0.1 * np.random.randn() for i in range(epochs)]
    val_losses = [2.3 - i * 0.35 + 0.15 * np.random.randn() for i in range(epochs)]
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}:")
        print(f"  Train loss: {train_losses[epoch]:.4f}")
        print(f"  Val loss: {val_losses[epoch]:.4f}")
    
    print("\n5. Saving best checkpoint...")
    print("6. Training complete!")
    
    return train_losses, val_losses

In [ ]:
import numpy as np
train_losses, val_losses = simulate_training_process()

In [ ]:
# Plot simulated training process
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', label='Training Loss')
plt.plot(range(1, len(val_losses) + 1), val_losses, 'r-o', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Simulated Training Process')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6단계: 모델 평가

훈련 후 모델을 평가해야 합니다:

In [ ]:
print("Model evaluation metrics:")
print("-" * 40)
print("Perplexity: Exponentiated average negative log-likelihood per token")
print("Loss: Negative log-likelihood loss on held-out data")
print("-" * 40)

In [ ]:
# Simulated perplexity calculation
final_val_loss = val_losses[-1]
perplexity = np.exp(final_val_loss)
print(f"Final validation perplexity: {perplexity:.4f}")

## 7단계: 훈련된 모델 사용

훈련이 완료되면 모델을 로드하여 사용할 수 있습니다:

In [ ]:
print("Code to load your trained model:")
print("""
from lobster.model import LobsterPMLM

# Load model from checkpoint
model = LobsterPMLM.load_from_checkpoint("path/to/best/checkpoint.ckpt")
model.eval()  # Set to evaluation mode

# Use for inference
sequence = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGH"
tokens = model.tokenizer(sequence, return_tensors="pt")
embeddings = model.model(input_ids=tokens["input_ids"], 
                          attention_mask=tokens["attention_mask"])
""")

## 고급: 프로덕션 모델을 위한 훈련 전략

더 큰 모델과 데이터셋의 경우 다음 전략을 고려하십시오:

In [ ]:
print("\nAdvanced training strategies:")
print("-" * 60)
print("1. Gradient Accumulation: For training with effectively larger batch sizes")
print("   trainer.accumulate_grad_batches=4")
print()
print("2. Mixed Precision Training: For faster training and lower memory usage")
print("   trainer.precision=16")
print()
print("3. Distributed Training: For multi-GPU setups")
print("   trainer.strategy='ddp' trainer.devices=4")
print()
print("4. Checkpoint Resuming: To continue training from a checkpoint")
print("   trainer.resume_from_checkpoint='path/to/last/checkpoint.ckpt'")
print("-" * 60)

## 결론

이 튜토리얼에서는 다음을 다루었습니다:

1. LBSTER 모델 훈련을 위한 데이터 준비
2. Hydra를 사용한 구성
3. 모델 아키텍처 구성
4. 훈련 과정
5. 평가 지표
6. 훈련된 모델 사용
7. 고급 훈련 전략

실제 데이터로 최적의 결과를 얻으려면 일반적으로 다음이 필요합니다:
- 더 큰 데이터셋 (>100K 시퀀스)
- 더 많은 컴퓨팅 리소스
- 더 긴 훈련 시간 (모델 크기에 따라 며칠에서 몇 주)
- 신중한 하이퍼파라미터 튜닝

LBSTER 논문은 올바른 구성과 데이터 처리 파이프라인을 사용하면 24 GPU 시간 만에 효과적인 단백질 언어 모델을 훈련할 수 있음을 보여줍니다.